# Stage 4 Hybrid CLIP + Qwen Reporter

Clean Stage 4 run for the operation plan: detector context crops, frozen Qwen reporter fields, and a train-selected CLIP coarse classifier. The classifier is trained on clean Stage 3 train crops only, then applied to predicted Stage 4 crops. The output includes baseline Stage 4 metrics, hybrid Stage 4 metrics, and a compact archive.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import csv

import pandas as pd

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")
RUN_NAME = "stage4_detector_to_vlm_pred_val_hybrid_clip_qwen_pad030_kaggle"
BASELINE_RUN_NAME = "stage4_detector_to_vlm_pred_val_context_pad030_maxpix401k"
ARCHIVE_PATH = Path("/kaggle/working/stage4_deliverables_hybrid_clip_qwen_pad030_kaggle.tar.gz")

DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
DETECTOR_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-detector-assets/idid-detector-assets"),
    Path("/kaggle/input/idid-detector-assets/idid-detector-assets"),
    Path("/kaggle/input/idid-detector-assets"),
]

STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
CLIP_LOGREG_C = 10.0
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]

print("RUN_NAME:", RUN_NAME)
print("BASELINE_RUN_NAME:", BASELINE_RUN_NAME)


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path: Path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def pick_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("not found among: " + "; ".join(str(x) for x in candidates))

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required for this run")

DATA_ROOT = None
for root in DATA_ROOT_CANDIDATES:
    if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("stage3_regrouped_v2 train/val JSONL was not found")

DETECTOR_ROOT = None
for root in DETECTOR_ROOT_CANDIDATES:
    if (root / "data/processed/val/images").exists() and (root / "data/processed/val/annotations.json").exists():
        DETECTOR_ROOT = root
        break
if DETECTOR_ROOT is None:
    raise FileNotFoundError("detector assets were not found")

train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
val_jsonl_stage3 = DATA_ROOT / VAL_JSONL_REL
stage4_gt_jsonl = pick_existing([
    DETECTOR_ROOT / "analysis/stage4_gt_remapped.jsonl",
    val_jsonl_stage3,
])
detector_input_dir = DETECTOR_ROOT / "data/processed/val/images"
coco_json = DETECTOR_ROOT / "data/processed/val/annotations.json"
weights_path = DETECTOR_ROOT / "outputs/train/detector_baseline/best.pth"

print("DATA_ROOT:", DATA_ROOT)
print("DETECTOR_ROOT:", DETECTOR_ROOT)
print("train_jsonl:", train_jsonl)
print("stage4_gt_jsonl:", stage4_gt_jsonl)
print("detector_input_dir:", detector_input_dir)
print("weights_path:", weights_path)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])
sh([
    "python", "-m", "pip", "install", "-q",
    "transformers==4.57.1", "accelerate", "qwen-vl-utils", "pyyaml", "tabulate", "scikit-learn", "hydra-core", "omegaconf", "pycocotools",
], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
import yaml

cfg = {
    "version": 1,
    "name": BASELINE_RUN_NAME,
    "stage4": {
        "run_name": BASELINE_RUN_NAME,
        "split": "val",
        "output_root": str(REPO_DIR / "outputs/stage4"),
    },
    "detector": {
        "experiment": "detector_baseline",
        "input_dir": str(detector_input_dir),
        "config_path": str(REPO_DIR / "src/configs/infer.yaml"),
        "weights_path": str(weights_path),
        "conf_threshold": 0.05,
        "iou_threshold": 0.5,
        "max_detections_per_image": 100,
        "vis_samples": 8,
        "device": "auto",
    },
    "crop_export": {
        "coco_json": str(coco_json),
        "images_dir": str(detector_input_dir),
        "include_categories": None,
        "padding_ratio": 0.30,
        "manifest_name": "pred_manifest.jsonl",
        "summary_name": "pred_manifest_summary.json",
        "limit": None,
    },
    "vlm": {
        "backend_mode": "qwen_hf",
        "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "prompt_version": "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath",
        "stage3_runner_config": str(REPO_DIR / "configs/pipeline/stage3_vlm_gt_baseline.yaml"),
        "run_id": "stage4_pred_vlm",
        "qwen_hf": {"max_pixels": 401408},
    },
    "analysis": {
        "ground_truth_jsonl": str(stage4_gt_jsonl),
        "match_iou_threshold": 0.5,
        "good_crop_iou_threshold": 0.7,
    },
}
config_path = REPO_DIR / "configs/stage4_kaggle_hybrid_base.yaml"
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))


In [ ]:
# Run baseline detector -> context crops -> Qwen reporter -> Stage 4 eval.
# This is the long cell.
sh(["python", "scripts/run_stage4_detector_to_vlm.py", "--config", str(config_path)], cwd=REPO_DIR, check=True)

baseline_run_dir = REPO_DIR / "outputs/stage4" / BASELINE_RUN_NAME
pred_manifest = baseline_run_dir / "02_pred_crops/pred_manifest.jsonl"
qwen_run_dir = baseline_run_dir / "03_vlm_pred/stage4_pred_vlm"
detector_predictions = baseline_run_dir / "01_detector/predictions.json"
baseline_eval_dir = baseline_run_dir / "04_eval"
print("baseline metrics:", (baseline_eval_dir / "stage4_metrics.json").read_text(encoding="utf-8")[:1200])


In [ ]:
# Train the coarse classifier on train only, using the same selected setup from the clean CLIP branch.
import numpy as np
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import CLIPModel, CLIPProcessor

train_rows = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
pred_rows = load_jsonl(pred_manifest)
print("train rows:", len(train_rows), pd.Series([r["coarse_class"] for r in train_rows]).value_counts().to_dict())
print("pred crop rows:", len(pred_rows))

def resolve_stage3_crop(row):
    p = Path(row["crop_path"])
    if p.is_absolute() and p.exists(): return p
    candidates = [train_jsonl.parent / p, train_jsonl.parent.parent / p, DATA_ROOT / p]
    for c in candidates:
        if c.exists(): return c
    raise FileNotFoundError(row["crop_path"])

def resolve_pred_crop(row):
    p = Path(row["crop_path"])
    if p.is_absolute() and p.exists(): return p
    candidates = [pred_manifest.parent / p, pred_manifest.parent.parent / p, baseline_run_dir / "02_pred_crops" / p]
    for c in candidates:
        if c.exists(): return c
    raise FileNotFoundError(row["crop_path"])

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model.eval()

def embed_paths(paths, batch_size=32):
    feats=[]
    with torch.no_grad():
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            images=[Image.open(p).convert("RGB") for p in batch_paths]
            inputs=clip_processor(images=images, return_tensors="pt", padding=True).to(device)
            out=clip_model.get_image_features(**inputs)
            out=out / out.norm(dim=-1, keepdim=True)
            feats.append(out.detach().cpu().numpy())
            print("embedded", min(i+batch_size, len(paths)), "/", len(paths))
    return np.concatenate(feats, axis=0)

train_paths=[resolve_stage3_crop(r) for r in train_rows]
pred_paths=[resolve_pred_crop(r) for r in pred_rows]
X_train=embed_paths(train_paths)
y_train=np.array([r["coarse_class"] for r in train_rows])
X_pred=embed_paths(pred_paths)

clf = LogisticRegression(C=CLIP_LOGREG_C, class_weight="balanced", max_iter=2000, random_state=42, multi_class="auto")
clf.fit(X_train, y_train)
pred_cls = clf.predict(X_pred)
proba = clf.predict_proba(X_pred)
classes = list(clf.classes_)
maxp = proba.max(axis=1)

classifier_csv = baseline_run_dir / "06_hybrid_clip_qwen/classifier_pred_crops.csv"
classifier_csv.parent.mkdir(parents=True, exist_ok=True)
rows=[]
for row, pred, conf, probs in zip(pred_rows, pred_cls, maxp, proba):
    out={"record_id": row["record_id"], "pred_coarse_class": str(pred), "confidence": float(conf)}
    for cls, val in zip(classes, probs):
        out[f"prob_{cls}"] = float(val)
    rows.append(out)
pd.DataFrame(rows).to_csv(classifier_csv, index=False)
print("classifier_csv:", classifier_csv)
display(pd.DataFrame(rows).head())


In [ ]:
# Merge CLIP coarse prediction into Qwen structured reporter output.
qwen_pred_path = qwen_run_dir / "predictions_vlm_labels_v1.jsonl"
hybrid_run_dir = baseline_run_dir / "06_hybrid_clip_qwen/stage4_pred_vlm_hybrid_hard"
hybrid_run_dir.mkdir(parents=True, exist_ok=True)

qwen_preds = {r["record_id"]: r for r in load_jsonl(qwen_pred_path)}
clf_rows = {r["record_id"]: r for r in pd.read_csv(classifier_csv).to_dict("records")}
hybrid_preds=[]
decisions=[]
for rid, qrow in qwen_preds.items():
    h=dict(qrow)
    crow=clf_rows.get(rid)
    old=h.get("coarse_class")
    if crow is not None:
        new=str(crow["pred_coarse_class"])
        conf=float(crow.get("confidence", 0.0))
        h["coarse_class"] = new
        h.setdefault("annotator_notes", "")
        note = f"hybrid coarse_class from CLIP linear probe, confidence={conf:.4f}; Qwen reporter fields retained"
        h["annotator_notes"] = (str(h.get("annotator_notes") or "") + " " + note).strip()
        decisions.append({"record_id": rid, "qwen_class": old, "classifier_class": new, "hybrid_class": new, "confidence": conf, "changed": old != new})
    else:
        decisions.append({"record_id": rid, "qwen_class": old, "classifier_class": "", "hybrid_class": old, "confidence": "", "changed": False})
    hybrid_preds.append(h)

write_jsonl(hybrid_run_dir / "predictions_vlm_labels_v1.jsonl", hybrid_preds)
# copy stage3 sidecars so run directory remains inspectable
for name in ["sample_results.jsonl", "raw_responses.jsonl", "parsed_predictions.jsonl", "run_summary.json"]:
    src=qwen_run_dir/name
    if src.exists(): shutil.copy2(src, hybrid_run_dir/name)
pd.DataFrame(decisions).to_csv(hybrid_run_dir / "hybrid_decisions.csv", index=False)
print("hybrid_run_dir:", hybrid_run_dir)
display(pd.DataFrame(decisions).head())


In [ ]:
# Validate and evaluate Stage 4 hybrid.
sh(["python", "scripts/validate_vlm_labels_v1.py", "--input", str(hybrid_run_dir / "predictions_vlm_labels_v1.jsonl")], cwd=REPO_DIR, check=True)

hybrid_eval_dir = baseline_run_dir / "06_hybrid_clip_qwen/eval_stage4_hybrid_hard"
sh([
    "python", "scripts/eval_stage4_detector_to_vlm.py",
    "--gt-jsonl", str(stage4_gt_jsonl),
    "--pred-manifest-jsonl", str(pred_manifest),
    "--pred-vlm-run-dir", str(hybrid_run_dir),
    "--detector-predictions-json", str(detector_predictions),
    "--coco-json", str(coco_json),
    "--match-iou-threshold", "0.5",
    "--good-crop-iou-threshold", "0.7",
    "--output-dir", str(hybrid_eval_dir),
], cwd=REPO_DIR, check=True)

baseline_metrics = read_json(baseline_eval_dir / "stage4_metrics.json")
hybrid_metrics = read_json(hybrid_eval_dir / "stage4_metrics.json")
comparison = {
    "baseline_pipeline_correct": baseline_metrics["counts"]["pipeline_correct_total"],
    "baseline_pipeline_rate": baseline_metrics["rates"]["pipeline_correct_rate"],
    "hybrid_pipeline_correct": hybrid_metrics["counts"]["pipeline_correct_total"],
    "hybrid_pipeline_rate": hybrid_metrics["rates"]["pipeline_correct_rate"],
    "delta_correct": hybrid_metrics["counts"]["pipeline_correct_total"] - baseline_metrics["counts"]["pipeline_correct_total"],
    "delta_rate": hybrid_metrics["rates"]["pipeline_correct_rate"] - baseline_metrics["rates"]["pipeline_correct_rate"],
    "baseline_vlm_correct_good": baseline_metrics["counts"]["vlm_correct_on_good_pred_crop_total"],
    "hybrid_vlm_correct_good": hybrid_metrics["counts"]["vlm_correct_on_good_pred_crop_total"],
}
comparison_path = baseline_run_dir / "06_hybrid_clip_qwen/stage4_hybrid_comparison.json"
comparison_path.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(json.dumps(comparison, indent=2))
display(pd.DataFrame([comparison]))


In [ ]:
# Compact report and archive.
report_dir = baseline_run_dir / "06_hybrid_clip_qwen"
base_cases = pd.read_csv(baseline_eval_dir / "stage4_case_table.csv")
hyb_cases = pd.read_csv(hybrid_eval_dir / "stage4_case_table.csv")
case_cmp = base_cases[["record_id", "gt_coarse_class", "pred_vlm_coarse_class", "error_bucket"]].rename(columns={"pred_vlm_coarse_class":"baseline_pred", "error_bucket":"baseline_bucket"}).merge(
    hyb_cases[["record_id", "pred_vlm_coarse_class", "error_bucket"]].rename(columns={"pred_vlm_coarse_class":"hybrid_pred", "error_bucket":"hybrid_bucket"}),
    on="record_id", how="outer"
)
case_cmp["changed_pred"] = case_cmp["baseline_pred"] != case_cmp["hybrid_pred"]
case_cmp["changed_bucket"] = case_cmp["baseline_bucket"] != case_cmp["hybrid_bucket"]
case_cmp.to_csv(report_dir / "stage4_hybrid_case_comparison.csv", index=False)

lines=[
    "# Stage 4 Hybrid CLIP + Qwen Reporter",
    "",
    f"Baseline run: `{BASELINE_RUN_NAME}`",
    f"Hybrid run: `{RUN_NAME}`",
    f"CLIP model: `{CLIP_MODEL_ID}`",
    f"Classifier: LogisticRegression C={CLIP_LOGREG_C}, class_weight=balanced, train only",
    "",
    "## Metrics",
    pd.DataFrame([comparison]).to_markdown(index=False),
    "",
    "## Error bucket comparison",
    "Baseline:",
    pd.Series(read_json(baseline_eval_dir / "stage4_error_breakdown.json")["counts"]).to_markdown(),
    "",
    "Hybrid:",
    pd.Series(read_json(hybrid_eval_dir / "stage4_error_breakdown.json")["counts"]).to_markdown(),
    "",
    "## Changed cases",
    case_cmp[case_cmp["changed_bucket"] | case_cmp["changed_pred"]].head(40).to_markdown(index=False),
]
(report_dir / "stage4_hybrid_report.md").write_text("\n".join(lines), encoding="utf-8")
print((report_dir / "stage4_hybrid_report.md").read_text(encoding="utf-8")[:3000])

package_root = REPO_DIR / "outputs/stage4_hybrid_clip_qwen_package"
if package_root.exists(): shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
for rel in [
    "04_eval/stage4_metrics.json",
    "04_eval/stage4_error_breakdown.json",
    "04_eval/stage4_case_table.csv",
    "04_eval/stage4_summary.md",
    "06_hybrid_clip_qwen/classifier_pred_crops.csv",
    "06_hybrid_clip_qwen/stage4_pred_vlm_hybrid_hard/predictions_vlm_labels_v1.jsonl",
    "06_hybrid_clip_qwen/stage4_pred_vlm_hybrid_hard/hybrid_decisions.csv",
    "06_hybrid_clip_qwen/eval_stage4_hybrid_hard/stage4_metrics.json",
    "06_hybrid_clip_qwen/eval_stage4_hybrid_hard/stage4_error_breakdown.json",
    "06_hybrid_clip_qwen/eval_stage4_hybrid_hard/stage4_case_table.csv",
    "06_hybrid_clip_qwen/stage4_hybrid_comparison.json",
    "06_hybrid_clip_qwen/stage4_hybrid_case_comparison.csv",
    "06_hybrid_clip_qwen/stage4_hybrid_report.md",
]:
    src=baseline_run_dir/rel
    if src.exists():
        dst=package_root/rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src,dst)
if ARCHIVE_PATH.exists(): ARCHIVE_PATH.unlink()
sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(package_root.parent), package_root.name], check=True)
print("Archive:", ARCHIVE_PATH)
shutil.rmtree(REPO_DIR, ignore_errors=True)
print("Cleaned work repo:", REPO_DIR)
